# عين | Ain
## AI-Powered Municipal Infrastructure Assistant

This notebook runs the complete project in Google Colab:

1. Installs the required packages.
2. Connects to OpenRouter securely.
3. Creates the two CrewAI agents.
4. Optionally connects to Google Sheets.
5. Launches a public Gradio interface.


## 1. Install packages

Run the next cell once. Colab may ask you to restart the runtime after installation.


In [ ]:
!pip install -q crewai gradio gspread google-auth


## 2. Import libraries and enter the OpenRouter key

The key is requested securely during runtime and is not saved in the notebook.


In [ ]:
import os
import re
import json
from getpass import getpass

from crewai import Agent, Task, Crew, Process, LLM
import gradio as gr

if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("Enter your OpenRouter API key: ")

llm = LLM(
    model="openrouter/openai/gpt-4o-mini",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    temperature=0
)

print("OpenRouter configured successfully.")


## 3. Create the AI agents


In [ ]:
report_analyzer = Agent(
    role="Infrastructure Report Analyzer",
    goal=(
        "Extract the issue type, location, and description from a citizen "
        "municipal infrastructure report without inventing missing details."
    ),
    backstory=(
        "You are a careful municipal report analyst. You only use information "
        "explicitly stated in the citizen report."
    ),
    llm=llm,
    verbose=False,
    allow_delegation=False
)

quality_reviewer = Agent(
    role="Infrastructure Quality Reviewer",
    goal=(
        "Review the extracted report, assign a priority, explain the reason, "
        "estimate confidence, and return valid JSON."
    ),
    backstory=(
        "You are a municipal quality reviewer. You check accuracy and completeness "
        "and never invent unsupported details."
    ),
    llm=llm,
    verbose=False,
    allow_delegation=False
)

print("Agents created successfully.")


## 4. Create the tasks and crew


In [ ]:
analysis_task = Task(
    description=(
        "Analyze this citizen report:\n\n{citizen_report}\n\n"
        "Extract only the issue_type, location, and description. "
        "Use null when information is missing."
    ),
    expected_output=(
        "A concise JSON object with issue_type, location, and description."
    ),
    agent=report_analyzer
)

review_task = Task(
    description=(
        "Review the analyzer output and the original citizen report. "
        "Return ONLY valid JSON with exactly these keys: "
        "issue_type, location, description, priority, priority_reason, "
        "confidence_score, status. "
        "Priority must be Low, Medium, or High. "
        "Confidence score must be an integer from 0 to 100. "
        "Status must be Ready for Submission or Needs More Information."
    ),
    expected_output=(
        "Valid JSON only, with the seven required keys and no markdown fences."
    ),
    agent=quality_reviewer,
    context=[analysis_task]
)

ain_crew = Crew(
    agents=[report_analyzer, quality_reviewer],
    tasks=[analysis_task, review_task],
    process=Process.sequential,
    verbose=False
)

print("Ain crew created successfully.")


## 5. Connect Google Sheets

Run the next cell if you want reports saved automatically.

The cell authenticates your Google account and creates a spreadsheet named
**Ain Infrastructure Reports** if it does not already exist.


In [ ]:
sheet = None

try:
    import gspread
    from google.colab import auth
    from google.auth import default
    from gspread.exceptions import SpreadsheetNotFound

    auth.authenticate_user()
    creds, _ = default()
    gc = gspread.authorize(creds)

    spreadsheet_name = "Ain Infrastructure Reports"

    try:
        spreadsheet = gc.open(spreadsheet_name)
    except SpreadsheetNotFound:
        spreadsheet = gc.create(spreadsheet_name)

    sheet = spreadsheet.sheet1

    headers = [
        "Issue Type",
        "Location",
        "Description",
        "Priority",
        "Priority Reason",
        "Confidence Score",
        "Status"
    ]

    if not sheet.get_all_values():
        sheet.append_row(headers)

    print("Google Sheet connected successfully:")
    print(spreadsheet.url)

except Exception as error:
    sheet = None
    print("Google Sheets was not connected.")
    print("The Gradio app can still run without saving reports.")
    print("Reason:", error)


## 6. Processing function


In [ ]:
def extract_json(text):
    text = text.strip()

    # Remove optional markdown code fences.
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s*```$", "", text)

    # Keep the first complete JSON object if extra text appears.
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError("The model response did not contain a JSON object.")

    return json.loads(match.group(0))


async def process_report(citizen_report):
    if not citizen_report or not citizen_report.strip():
        return json.dumps(
            {"error": "Please enter a citizen report."},
            ensure_ascii=False,
            indent=2
        )

    try:
        result = await ain_crew.kickoff_async(
            inputs={"citizen_report": citizen_report.strip()}
        )

        data = extract_json(result.raw)

        required_keys = [
            "issue_type",
            "location",
            "description",
            "priority",
            "priority_reason",
            "confidence_score",
            "status"
        ]

        for key in required_keys:
            data.setdefault(key, None)

        if sheet is not None:
            sheet.append_row([
                data["issue_type"],
                data["location"],
                data["description"],
                data["priority"],
                data["priority_reason"],
                data["confidence_score"],
                data["status"]
            ])

        return json.dumps(data, ensure_ascii=False, indent=2)

    except Exception as error:
        return json.dumps(
            {
                "error": "The report could not be processed.",
                "details": str(error)
            },
            ensure_ascii=False,
            indent=2
        )


## 7. Launch Gradio

Run the final cell. A public Gradio link ending in `.gradio.live` will appear.

That link is temporary and remains active only while this Colab runtime is running.


In [ ]:
custom_css = '''
.gradio-container {
    max-width: 980px !important;
    margin: auto !important;
}
'''

demo = gr.Interface(
    fn=process_report,
    inputs=gr.Textbox(
        lines=7,
        label="Citizen Report",
        placeholder="Example: There is a large pothole near a school in Al Narjis District, Riyadh."
    ),
    outputs=gr.Code(
        language="json",
        label="Validated Report"
    ),
    title="عين | Ain",
    description=(
        "AI-Powered Municipal Infrastructure Assistant — "
        "Analyze, validate, and prioritize citizen infrastructure reports."
    ),
    examples=[
        ["There is a large pothole near a school in Al Narjis District, Riyadh."],
        ["Several streetlights are not working on King Fahd Road."]
    ],
    css=custom_css
)

demo.launch(share=True, debug=True)
